## Transformer based character model

We will first define a transformer piece by piece

In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
len(text)

1115394

In [4]:
# find all unique characters in text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
vocab_size


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [9]:
# create mapping from char <-> int
stoi = { ch:i for i, ch in enumerate(chars)}
itos = { i:ch for i, ch in enumerate(chars)}
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

print(encode('hey there'))
print(decode(encode('hey there')))

[46, 43, 63, 1, 58, 46, 43, 56, 43]
hey there


In [ ]:
# encode entirety of text and wrap in tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long) # int64 type

In [12]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

We will train transformer with randomly sample chunks (blocks) at a time, since training on entirety of data is expensive

In [16]:
# toy example
block_size = 8 # max context length

x = train_data[:block_size]
y = train_data[1:block_size+1] # next char after x aka target for x
for t in range(block_size):
    context = x[:t+1] # up to t and including t
    target = y[t]

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # number of indept sequences to process in parallel = B
block_size = 8 # max context length = T

def get_batch(split):
    data = train_data if split == ' train' else val_data
    # generate batch size numbers (4) of values between 0 and len(data) - block_size, which are indexes to grab from
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # torch.stack combines all acquired data into 1 tensor
    # initially, 4 row tensors of length 8, becomes 4x8 tensor for both x and y
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)


torch.Size([4, 8])
tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])
torch.Size([4, 8])
tensor([[ 1, 52, 53, 58,  1, 58, 47, 50],
        [ 1, 54, 50, 39, 52, 58, 43, 58],
        [58, 46, 47, 57,  1, 50, 47, 60],
        [32, 46, 43, 56, 43,  1, 42, 53]])


## Bigram language model

In [17]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [ ]:
n_embd = 32

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # embedding table of probabilities of next char given current char
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each posn from 1 to blcok_size-1 gets embedding vector
        self.lm_head = nn.Linear(n_embd, vocab_size) # linear layer
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C) tensor
        # B = 4, T = 8, C = vocab_size = 65

        pos_emb = self.position_embedding_table(torch.arange(T)) #(T, C)
        x = tok_emb + pos_emb # (B, T, C), holds token identities AND posn at which tokens occur
        logits = self.lm_head(tok_emb) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            # reshape to match torch implementation of cross_entropy
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # each row corresponds to one prediction
            targets = targets.view(B*T)


            # expects inputs: (N, C), targte: (N)
            # N = number of predictions
            # C = number of classes
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # predictions at current indices
            logits, loss = self(idx)

            # pluck out last time step to get a (B, C)
            logits = logits[:, -1, :]

            # softmax to get prob
            probs = F.softmax(logits, dim=-1)

            # sample from distn
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # concat sampled index to running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
        
m = BigramLanguageModel()
logits, loss = m(xb, yb)
loss

idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


Cj'fKP&XWTEc:f-V;ijn lp$MHppHLiWdW:AAuJCBoMj.C$tI$LoaoSxO:N$t;aUpGRjC-Phyr$$:I-Pb'EA&gf.dDJU3f?'EXrW


Train model

In [28]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [33]:
batch_size = 32
for steps in range(10000):
    # sample batch of data
    xb, yb = get_batch('train')

    # loss evaluation
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.283473014831543


In [34]:
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))



Who ird s, de bre.
KAnd I pps pra host mareme prinndem w fis h br, shetare: d selll for fobo I y ha


### Math of self-attention

We want tokens to talk each other, such that a token only gets info from ones before it

In [35]:
B, T, C= 4, 8, 2
x = torch.randn(B,T,C)

In [36]:
# toy implementation: x[b, t] = mean_(i<=t) x[b, i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)

In [39]:
print(x[0])
xbow[0]
# each token is the ave of itself and all previous tokens

tensor([[ 0.1737, -0.1425],
        [ 2.0987,  0.0198],
        [-0.5152, -1.5360],
        [ 0.0102,  0.7225],
        [ 0.7807,  0.5716],
        [ 0.0164,  0.1316],
        [ 1.8871, -0.3909],
        [-0.3973,  0.2154]])


tensor([[ 0.1737, -0.1425],
        [ 1.1362, -0.0613],
        [ 0.5858, -0.5529],
        [ 0.4419, -0.2340],
        [ 0.5096, -0.0729],
        [ 0.4274, -0.0388],
        [ 0.6360, -0.0891],
        [ 0.5068, -0.0510]])

We can make above very efficient via mat mul

In [45]:
# make lower triangular matrix
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
a
# a @ b now returns a matrix where the columns are a rolling average

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [ ]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
wei
xbow2 = wei @ x # (T, T) @ (B, T, C) broadcast to (B, T, T) @ (B, T, C) --> for each batch B, (T, T) @ (T, C) so we end up with (B, T, C)
xbow[0], xbow2[0] # should be identical, but xbow2 is more efficient

(tensor([[ 0.1737, -0.1425],
         [ 1.1362, -0.0613],
         [ 0.5858, -0.5529],
         [ 0.4419, -0.2340],
         [ 0.5096, -0.0729],
         [ 0.4274, -0.0388],
         [ 0.6360, -0.0891],
         [ 0.5068, -0.0510]]),
 tensor([[ 0.1737, -0.1425],
         [ 1.1362, -0.0613],
         [ 0.5858, -0.5529],
         [ 0.4419, -0.2340],
         [ 0.5096, -0.0729],
         [ 0.4274, -0.0388],
         [ 0.6360, -0.0891],
         [ 0.5068, -0.0510]]))

Version 3 which uses softmax:

In [57]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei =wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

## Self-attention

In practice, wei is determined by data and depends on the relationship between words, which drives self attention. At each position, there exists two vectors: query and key

Query: what am I looking for?

Key: what do I contain?

Dot product between Q and K becomes wei. If Q, K aligned, the weight is greater in value.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
wei = q @ k.transpose(-2, -1) # transpose last 2 dim of k to get (B, 7, 16) @ (B, 16, T) --> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf')) # decoder block, prevents current word from talking to future
wei = F.softmax(wei, dim=-1) # can divide by head_size ** 0.5 to control variance at initialization

v = value(x)
out = wei @ v

wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Value: how much info is pulled from that position and carried forward when attended to?

Head: each performs in lower dim space to potentially encode a different relationship

Multihead attention: multiple heads in parallel and concatinate results. Each indivially has smaller dimension

Transformer block: repeated blocks of (communication followed by computation). Communication from attention, computation from adding of residual connection and layernorm. Then go through feed forward network (linear layer then ReLU gate)

In these blocks, we want to add residual connections st we add that initial data to the final result from the layers

Layer norm: similar to batchNorm (which normalizes the distn of each feature across a batch to N(0, 1)), but we normalize each row instead of column for layer norm (normalization of all features of a single input from batch). This makes so that no single feature dominiates for each token and that it is well scaled no matter what batch it is in.

Drop out: helps to regularize and prevent overfitting

## Pretraining Summary

B = batch size = num rows

T = max context length = num cols

One training batch is then (B, T) and contains encoded text (tokens) and special tokens that signal end of phrases. Each token has all the tokens before it in its row as context and the next token as its target.